<a href="https://colab.research.google.com/github/Mengteth/AI-Project/blob/main/Mid_Term.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.4/718.4 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.4/

In [11]:
# ==============================
# Step 0: Install required packages
# ==============================
!pip install mlflow scikit-learn pandas

# ==============================
# Step 1: Imports
# ==============================
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import shutil

# ==============================
# Step 2: Set MLflow tracking URI and experiment
# ==============================
mlflow.set_tracking_uri("/content/mlruns")  # save locally in Colab
mlflow.set_experiment("Breast_Cancer_Classification")

# ==============================
# Step 3: Load dataset and split
# ==============================
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features for Logistic Regression and SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==============================
# Step 4: Function to train & log model
# ==============================
def train_and_log_model(model, model_name, X_train, X_test, y_train, y_test, **params):
    with mlflow.start_run(run_name=model_name):
        # Log hyperparameters
        mlflow.log_params(params)

        # Train model
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)

        # Compute metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Log metrics
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)

        # Log model with input example to remove warning
        mlflow.sklearn.log_model(
            model,
            artifact_path="model",
            input_example=pd.DataFrame(X_test[:5], columns=data.feature_names)
        )

        # Print summary
        print(f"✅ {model_name} logged")
        print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1 Score: {f1:.4f}\n")

# ==============================
# Step 5: Train & log 3 models
# ==============================
# Logistic Regression
train_and_log_model(
    LogisticRegression(max_iter=5000),
    "Logistic Regression",
    X_train_scaled, X_test_scaled, y_train, y_test,
    max_iter=5000, solver='lbfgs'
)

# SVM
train_and_log_model(
    SVC(kernel='rbf', probability=True),
    "SVM",
    X_train_scaled, X_test_scaled, y_train, y_test,
    kernel='rbf', C=1.0
)

# Random Forest
train_and_log_model(
    RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42),
    "Random Forest",
    X_train, X_test, y_train, y_test,
    n_estimators=100, max_depth=None
)

# ==============================
# Step 6: Compare all 3 models
# ==============================
client = MlflowClient()
experiment = mlflow.get_experiment_by_name("Breast_Cancer_Classification")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

summary = []
for r in runs:
    summary.append({
        "Model": r.data.tags.get("mlflow.runName"),
        "Accuracy": r.data.metrics.get("accuracy"),
        "Precision": r.data.metrics.get("precision"),
        "Recall": r.data.metrics.get("recall"),
        "F1 Score": r.data.metrics.get("f1_score")
    })

df_summary = pd.DataFrame(summary)
df_summary = df_summary.sort_values(by="Accuracy", ascending=False)

print("📊 Comparison of all 3 models:")
df_summary

# ==============================
# Step 7: Optional: Save MLflow logs as ZIP
# ==============================
shutil.make_archive("mlruns_backup", 'zip', "/content/mlruns/")
print("🎉 MLflow logs and models saved as mlruns_backup.zip")


2025/10/06 13:54:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-value

✅ Logistic Regression logged
Accuracy: 0.9737, Precision: 0.9722, Recall: 0.9859, F1 Score: 0.9790



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


✅ SVM logged
Accuracy: 0.9825, Precision: 0.9726, Recall: 1.0000, F1 Score: 0.9861



2025/10/06 13:54:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


✅ Random Forest logged
Accuracy: 0.9649, Precision: 0.9589, Recall: 0.9859, F1 Score: 0.9722

📊 Comparison of all 3 models:
🎉 MLflow logs and models saved as mlruns_backup.zip


In [12]:
!pip install joblib


In [13]:
# Sort by Accuracy (or F1 Score)
best_model_name = df_summary.sort_values(by="Accuracy", ascending=False).iloc[0]["Model"]
print(f"🏆 Best model based on Accuracy: {best_model_name}")


🏆 Best model based on Accuracy: SVM


In [15]:
# Retrieve the best model based on the run information from MLflow
client = MlflowClient()
experiment = mlflow.get_experiment_by_name("Breast_Cancer_Classification")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

# Find the run corresponding to the best model name
best_run = None
for r in runs:
    if r.data.tags.get("mlflow.runName") == best_model_name:
        best_run = r
        break

if best_run:
    # Load the best model
    best_model_uri = f"runs:/{best_run.info.run_id}/model"
    best_model = mlflow.sklearn.load_model(best_model_uri)
    print(f"🏆 Successfully loaded the best model: {best_model_name}")
else:
    print(f"Could not find run for best model: {best_model_name}")
    best_model = None

🏆 Successfully loaded the best model: SVM


In [16]:
import joblib

joblib.dump(best_model, "/content/best_model.pkl")
print("✅ Best model saved as best_model.pkl")


✅ Best model saved as best_model.pkl


In [17]:
from google.colab import files

files.download("/content/best_model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
!pip install fastapi uvicorn joblib


In [19]:
# app.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

# Load the saved best model
model = joblib.load("/content/best_model.pkl")

# Create FastAPI instance
app = FastAPI(title="Breast Cancer Prediction API")

# Define input data schema
class PatientData(BaseModel):
    mean_radius: float
    mean_texture: float
    mean_perimeter: float
    mean_area: float
    mean_smoothness: float
    # Add all 30 features from the dataset if needed
    # For simplicity, you can reduce features or use top ones

# Create a prediction endpoint
@app.post("/predict")
def predict(data: PatientData):
    # Convert input data to numpy array
    input_data = np.array([[v for v in data.dict().values()]])
    # Predict class
    prediction = model.predict(input_data)
    # Map 0/1 to labels
    label = "malignant" if prediction[0] == 0 else "benign"
    return {"prediction": label}


In [22]:
!pip install pyngrok


In [23]:
from pyngrok import ngrok

!ngrok authtoken 33hBXJa0l2okgDDtAa9UxgkGZ2o_87eNkVwXGtLK4JZN9mJqV


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [25]:
# Run FastAPI server
!nohup uvicorn app:app --host 0.0.0.0 --port 5000 &

# Create a public URL
public_url = ngrok.connect(5000)
print("🚀 FastAPI is live at:", public_url)


nohup: appending output to 'nohup.out'
🚀 FastAPI is live at: NgrokTunnel: "https://sizable-shelli-comatosely.ngrok-free.dev" -> "http://localhost:5000"
